# Explore K-Hairstyle



In [8]:
import json
from pathlib import Path

import pandas as pd

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'backend').exists() and (candidate / 'notebooks').exists():
            return candidate
    raise RuntimeError('Could not locate the project root from the current notebook session.')

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
DATASET_ROOT = PROJECT_ROOT / 'backend' / 'data' / 'raw' / 'khairstyle' / 'mqset'
IMAGE_ROOT = DATASET_ROOT / 'images' / 'images_mqset001'
LABEL_ROOT = DATASET_ROOT / 'labels' / 'labels_mqset'

print('Dataset root exists:', DATASET_ROOT.exists())
print('Image root exists:', IMAGE_ROOT.exists())
print('Label root exists:', LABEL_ROOT.exists())


Dataset root exists: True
Image root exists: True
Label root exists: True


In [9]:
json_paths = sorted(LABEL_ROOT.rglob('*.json'))
image_paths = sorted(IMAGE_ROOT.rglob('*.jpg'))

pd.Series(
    {
        'image_count': len(image_paths),
        'label_count': len(json_paths),
        'sample_image': image_paths[0].name if image_paths else None,
        'sample_label': json_paths[0].name if json_paths else None,
    }
)


image_count                  48252
label_count                  48252
sample_image     DSS268961-001.jpg
sample_label    DSS268961_001.json
dtype: object

In [10]:
if json_paths:
    sample_json_path = json_paths[0]
    sample_payload = json.loads(sample_json_path.read_text(encoding='utf-8', errors='replace'))
    sample_payload
else:
    sample_payload = {}
    print('No JSON labels were found under', LABEL_ROOT)


In [11]:
key_fields = [
    'basestyle',
    'basestyle-type',
    'length',
    'curl',
    'bang',
    'side',
    'color',
    'partition',
    'sex',
    'filename',
]

pd.Series({field: sample_payload.get(field) for field in key_fields})


basestyle                       가르마
basestyle-type                    단
length                           남자
curl                              X
bang                    기타(남자 내림머리)
side                            원블럭
color                            블랙
partition                       5:5
sex                               남
filename          DSS268961-001.jpg
dtype: object

In [12]:
records = []
for path in json_paths[:200]:
    payload = json.loads(path.read_text(encoding='utf-8', errors='replace'))
    records.append(
        {
            'basestyle': payload.get('basestyle'),
            'length': payload.get('length'),
            'curl': payload.get('curl'),
            'bang': payload.get('bang'),
            'side': payload.get('side'),
            'color': payload.get('color'),
            'sex': payload.get('sex'),
            'before_after': payload.get('before-after'),
        }
    )

sample_df = pd.DataFrame(records)
sample_df.head()


,basestyle,length,curl,bang,side,color,sex,before_after
0,가르마,남자,X,기타(남자 내림머리),원블럭,블랙,남,after
1,가르마,남자,X,기타(남자 내림머리),원블럭,블랙,남,after
2,가르마,남자,X,기타(남자 내림머리),원블럭,블랙,남,after
3,가르마,남자,X,기타(남자 내림머리),원블럭,블랙,남,after
4,가르마,남자,X,기타(남자 내림머리),원블럭,블랙,남,after


In [13]:
field_completeness = sample_df.notna().mean().sort_values(ascending=False)
field_completeness


basestyle       1.0
length          1.0
curl            1.0
bang            1.0
side            1.0
color           1.0
sex             1.0
before_after    1.0
dtype: float64

In [14]:
print('Top-level label folders:')
for path in sorted([folder for folder in LABEL_ROOT.iterdir() if folder.is_dir()])[:10]:
    print('-', path.name)

print('\nNote: some text values appear mojibake-encoded in the raw JSON. We should normalize encoding during preprocessing.')


Top-level label folders:
- 0002.mqset

Note: some text values appear mojibake-encoded in the raw JSON. We should normalize encoding during preprocessing.


## Takeaways

- `K-Hairstyle` carries rich hairstyle annotations that are useful for attribute learning.
- The raw JSON schema is broad, but Version 1 should only map the fields we need for hairstyle attributes.
- Some values appear encoding-damaged when read naively, so the preprocessing notebook should handle normalization explicitly.
